# TS pipeline

Fit one spine from mesh and skeleton through basis optimization, radius estimation, and validation.

## Imports

Load MaSCaF fitting tools and logging. Plots use the principal-axis camera:
the longest axis lies across the figure unless ``orientation="vertical"``.

In [ ]:
import logging
import os
from typing import TYPE_CHECKING

from mascaf import *
from mascaf.cable_fitting import _compute_morphology_node_radii
from swctools import SWCModel

if TYPE_CHECKING:
    import plotly.graph_objects as go  # type: ignore

logging.basicConfig(level=logging.INFO)

print("✅ Libraries imported successfully!")


✅ Libraries imported successfully!


## Load mesh and skeleton

Load the processed mesh and MCF skeleton for the chosen spine, then take `max_edge_length` and basis-optimizer options from the fit oracle. The figures are the mesh alone and the mesh with the skeleton, written under `viz/`.

In [2]:
spine_idx = 1
qst = 0.5
mcst = 5

pdf_scale = 2

mesh_path = f"../../data/mesh/processed/TS{spine_idx}.obj"
mm = MeshManager(mesh_path=mesh_path)
model_length = mm.bounding_box_diagonal()

polylines_name = f"TS{spine_idx}_qst{qst}_mcst{mcst}"
skeleton = SkeletonGraph.from_txt(
    f"../../data/mcf_skeletons/{polylines_name}.polylines.txt"
)

suggested = suggest_fit_parameters(
    mm,
    skeleton,
)
for line in suggested.rationale:
    print(f"Oracle: {line}")

params = suggested

mel_tag = f"{params.max_edge_length:0.2f}"

print(params)

INFO:mascaf.mesh:Loaded mesh: 2378 vertices, 4788 faces
INFO:mascaf.fit_oracle:Fit oracle: mel=133.768 (frac=0.04935, mel/t=1.716), n_rays=12, active_resample=[0.02468, 0.04935]×D


Oracle: thickness-based mel = 2 × SDF median (77.9434) → 155.887
Oracle: capped by skeleton sampling density: L/n_target = 8828.7/66 → 133.768 (mel/t=1.716)
Oracle: equivalent bbox fraction = 0.04935 (D=2710.45, D/t=34.77, L/t=113.3)
SuggestedFitParameters(max_edge_length=133.76822349551756, max_edge_length_fraction=0.04935276543605409, basis_optimizer_options=BasisOptimizerOptions(do_pruning=True, pruning_length=None, pruning_length_fraction=0.2, pruning_iterative=True, do_snapping=True, do_forcing=True, max_iterations=30, convergence_threshold=0.0001, preserve_terminal_nodes=True, preserve_branch_nodes=False, n_rays=12, ray_jitter=0.1, alpha_s=0.1, step_scale=0.5, repulsion_power=1.0, localization_beta=2.0, weight_epsilon=1e-06, step_cap_factor=0.5, outside_distance_tol=None, outside_distance_tol_fraction=1e-06, centering_error_stop_fraction=0.1, centering_error_plateau_tol=0.001, centering_error_plateau_patience=2, centering_error_increase_patience=2, snap_ray_perturb_scales=(0.0001

In [3]:

fig_out_dir = f"../../viz/ts{spine_idx}"
os.makedirs(fig_out_dir, exist_ok=True)

mesh_fig: "go.Figure"
mesh_fig, camera = visualize_mesh_3d(
    mm,
    skel=None,
    show_axes=False,
    title="",
    orientation="horizontal",
    return_camera=True,
    eye_scale=1.0,
)
mesh_fig.write_image(
    f"{fig_out_dir}/TS{spine_idx}_mesh.pdf",
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
mesh_fig.show()

mesh_skel_fig: "go.Figure" = visualize_mesh_3d(
    mm,
    skel=skeleton,
    show_axes=False,
    title="",
    orientation="horizontal",
    camera=camera,
    # eye_scale=1.0,
)
mesh_skel_fig.write_image(
    f"{fig_out_dir}/TS{spine_idx}_mesh_skel.pdf",
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
mesh_skel_fig.show()

C:\Users\MainUser\AppData\Local\Temp\ipykernel_8916\2112416549.py:14: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  mesh_fig.write_image(
INFO:choreographer.browsers.chromium:Chromium init'ed with kwargs {}
INFO:choreographer.browsers.chromium:Found chromium path: C:\Program Files (x86)\Google\Chrome\Application\chrome.exe
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmpj8r_b2yp.
INFO:choreographer.browser_async:Opening browser.
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmpgrk2yt2_.
INFO:choreographer.browsers.chromium:Temporary directory at: C:\Users\MainUser\AppData\Local\Temp\tmpgrk2yt2_
INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmpj8r_b2yp/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido.kalei

C:\Users\MainUser\AppData\Local\Temp\ipykernel_8916\2112416549.py:33: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  mesh_skel_fig.write_image(
INFO:choreographer.browsers.chromium:Chromium init'ed with kwargs {}
INFO:choreographer.browsers.chromium:Found chromium path: C:\Program Files (x86)\Google\Chrome\Application\chrome.exe
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmp16p9_rot.
INFO:choreographer.browser_async:Opening browser.
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmp42kbd035.
INFO:choreographer.browsers.chromium:Temporary directory at: C:\Users\MainUser\AppData\Local\Temp\tmp42kbd035
INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmp16p9_rot/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido.

## Basis optimization

Resample the skeleton into a morphology basis and move nodes with `BasisOptimizer` before any radius fitting. The figure overlays the original basis (red) and the optimized basis (blue).

In [4]:
print(params.basis_optimizer_options)

BasisOptimizerOptions(do_pruning=True, pruning_length=None, pruning_length_fraction=0.2, pruning_iterative=True, do_snapping=True, do_forcing=True, max_iterations=30, convergence_threshold=0.0001, preserve_terminal_nodes=True, preserve_branch_nodes=False, n_rays=12, ray_jitter=0.1, alpha_s=0.1, step_scale=0.5, repulsion_power=1.0, localization_beta=2.0, weight_epsilon=1e-06, step_cap_factor=0.5, outside_distance_tol=None, outside_distance_tol_fraction=1e-06, centering_error_stop_fraction=0.1, centering_error_plateau_tol=0.001, centering_error_plateau_patience=2, centering_error_increase_patience=2, snap_ray_perturb_scales=(0.0001, 0.001, 0.01, 0.05), snap_ray_perturb_angles=8, snap_chord_fraction=0.25, snap_min_chord_length=None, snap_min_chord_fraction=0.0005, active_resample=True, active_resample_min_fraction=0.024676382718027044, active_resample_max_fraction=0.04935276543605409, active_resample_allow_cycle_collapse=False)


In [5]:
basis = MorphologyGraph.from_skeleton_graph_resample(
    skeleton,
    float(params.max_edge_length),
)
print(
    f"Initial basis: {basis.number_of_nodes()} nodes, "
    f"{basis.number_of_edges()} edges"
)

optimizer = BasisOptimizer(basis, mm.mesh, params.basis_optimizer_options)
optimized_basis = optimizer.optimize()
stats = optimizer.get_optimization_stats()
print("Basis optimization statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

basis_opt_fig: "go.Figure" = visualize_mesh_3d(
    mm,
    skel=[basis, optimized_basis],
    show_axes=False,
    title="",
    orientation="horizontal",
    camera=camera,
    skel_color=["red", "blue"],
    skel_line_width=3.0,
    skel_marker_size=2.0,
)
basis_opt_fig.write_image(
    f"{fig_out_dir}/TS{spine_idx}_mel{mel_tag}_basis_opt.pdf",
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
basis_opt_fig.show()

INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:mascaf.morphology_graph:Resampled morphology: 88 nodes, 94 edges (max_edge_length=133.7682, source 376 nodes / 383 edges, 49 sections)
INFO:mascaf.basis_optimizer:Starting basis optimization...
INFO:mascaf.basis_optimizer:  Input: 88 nodes, 94 edges, total length 8563.1565
INFO:mascaf.basis_optimizer:Phase 0 - Pruning
INFO:mascaf.basis_optimizer:  Removing terminal stubs with length < 85.7676
INFO:mascaf.basis_optimizer:  Pruning done: 88 → 79 nodes (1 rounds)
INFO:mascaf.basis_optimizer:Phase 1 - Snapping: 6 nodes outside mesh


Initial basis: 88 nodes, 94 edges


INFO:mascaf.basis_optimizer:Phase 2 - Forcing: max 30 iterations
INFO:mascaf.basis_optimizer:  Forcing alpha_s=0.1000 step_scale=0.5000 ray_jitter=0.1000 active_resample=True
INFO:mascaf.basis_optimizer:  Active resample edge lengths in [6.688411e+01, 1.337682e+02] (fractions of bbox diagonal)
INFO:mascaf.basis_optimizer:  Iteration 0: active resample merged=2 split=0
INFO:mascaf.basis_optimizer:  Iteration 0: avg movement = 4.443444, centering_error = 8.090327e+02
INFO:mascaf.basis_optimizer:  Iteration 1: active resample merged=1 split=0
INFO:mascaf.basis_optimizer:  Iteration 1: avg movement = 3.493445, centering_error = 6.325662e+02
INFO:mascaf.basis_optimizer:  Iteration 2: active resample merged=1 split=2
INFO:mascaf.basis_optimizer:  Iteration 2: avg movement = 3.307931, centering_error = 5.888234e+02
INFO:mascaf.basis_optimizer:  Iteration 3: active resample merged=1 split=3
INFO:mascaf.basis_optimizer:  Iteration 3: avg movement = 2.777267, centering_error = 4.952483e+02
INFO:

Basis optimization statistics:
  num_nodes: 86
  num_edges: 92
  num_terminal_nodes: 9
  num_branch_nodes: 14
  total_length: 8735.869433058138
  nodes_outside_mesh: 0


INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmpgep4qwyk/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido.kaleido:Got ECE7
INFO:kaleido.kaleido:Reloading tab ECE7 before return.
INFO:kaleido.kaleido:Putting tab ECE7 back (queue size: 0).
INFO:kaleido.kaleido:Waiting for all cleanups to finish.
INFO:kaleido.kaleido:Exiting Kaleido.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:kaleido.kaleido:Cancelling tasks.
INFO:kaleido.kaleido:Exiting Kaleido/Choreo.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:kaleido.kaleido:Cancelling tasks.

## Cable fitting

Estimate node radii on the optimized basis with the equivalent-area strategy, write the SWC. The figure is the fitted morphology.

In [6]:
swc_out_dir = f"../../data/swc/current/{polylines_name}"
swc_filepath = f"{swc_out_dir}/TS{spine_idx}_mel{mel_tag}.swc"

if not os.path.exists(swc_out_dir):
    os.makedirs(swc_out_dir)

fit_options = FitOptions(
    max_edge_length=params.max_edge_length,
    radius_strategy="equivalent_area",
    section_probe_eps=1e-4,
    section_probe_tries=3,
    multi_tangent_reduction="mean",
    basis_optimizer_options=None,
)
morph = optimized_basis.copy()
_compute_morphology_node_radii(morph, mm.mesh, fit_options)

morph.to_swc_file(swc_filepath)

model = SWCModel.from_swc_file(swc_filepath)
model.print_attributes(node_info=False, edge_info=False)
morph_fig = visualize_cable_3d(
    model,
    slider=False,
    title="",
    width=800,
    height=600,
    show_axes=False,
    orientation="horizontal",
    camera=camera,
)
morph_filename = (
    f"{fig_out_dir}/TS{spine_idx}_mel{mel_tag}_" f"morph.pdf"
)
morph_fig.write_image(
    morph_filename,
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
morph_fig.show()

INFO:mascaf.cable_fitting:Computing node radii: strategy=equivalent_area, 86 nodes, probe_eps=1.00e-04, tries=3
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:mascaf.cable_fitting:Radii computed: min=13.996763 max=197.182905 mean=65.032399 median=57.750119
INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=93 reconnections=7 header=11
INFO:swctools.model:SWCModel.from_parse_result records=93 reconnections=7 header=11
INFO:swctools.model:SWCModel.from_swc_file built nodes=93 edges=92 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=92 sides=16 end_caps=False verts=2944 faces=2944
INFO:swctools.geometry:FrustaSet.from_swc_model edges=92 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=92 show_frusta=True show_centroid=True


SWCModel: nodes=93, edges=92, components=1, cycles=0, branch_points=11, roots=1, leaves=15, self_loops=0, density=0.0215


C:\Users\MainUser\AppData\Local\Temp\ipykernel_8916\1772379697.py:35: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  morph_fig.write_image(
INFO:choreographer.browsers.chromium:Chromium init'ed with kwargs {}
INFO:choreographer.browsers.chromium:Found chromium path: C:\Program Files (x86)\Google\Chrome\Application\chrome.exe
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmp00uy_24l.
INFO:choreographer.browser_async:Opening browser.
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmpdm24xe5m.
INFO:choreographer.browsers.chromium:Temporary directory at: C:\Users\MainUser\AppData\Local\Temp\tmpdm24xe5m
INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmp00uy_24l/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido.kale

### Validation after cable fitting

In [7]:
validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 2378 vertices, 4788 faces
INFO:mascaf.validation:  Skeleton: 376 nodes, 383 edges
INFO:mascaf.validation:  MorphologyGraph: 86 nodes, 92 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       70083740.2341
INFO:mascaf.validation:---- Morphology volume: 151991470.4619
INFO:mascaf.validation:---- Ratio:             2.1687
INFO:mascaf.validation:---- Error:             81907730.2278
INFO:mascaf.validation:---- Relative error:    116.87%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         3124839.6923
INFO:mascaf.validation:---- Morphology area:   4254146.9075
INFO:mascaf.validation:---- Ratio:             1.3614
INFO:mascaf.validation:---- Error:             1129307.2152
INFO:mascaf.validation:---- Relative error:    36.14%
INFO:mascaf

{'n_outside_nodes': 0, 'n_crossing_edges': 10, 'n_fully_outside_edges': 0}

## Terminal extension

In [8]:
morph.extend_terminals(length_scale=0.5, radius_fraction=0.5)
validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 2378 vertices, 4788 faces
INFO:mascaf.validation:  Skeleton: 376 nodes, 383 edges
INFO:mascaf.validation:  MorphologyGraph: 95 nodes, 101 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       70083740.2341
INFO:mascaf.validation:---- Morphology volume: 153548674.5052
INFO:mascaf.validation:---- Ratio:             2.1909
INFO:mascaf.validation:---- Error:             83464934.2711
INFO:mascaf.validation:---- Relative error:    119.09%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         3124839.6923
INFO:mascaf.validation:---- Morphology area:   4275817.0398
INFO:mascaf.validation:---- Ratio:             1.3683
INFO:mascaf.validation:---- Error:             1150977.3475
INFO:mascaf.validation:---- Relative error:    36.83%
INFO:masca

{'n_outside_nodes': 0, 'n_crossing_edges': 10, 'n_fully_outside_edges': 0}

## Surface-area normalization

Scale radii so the morphology surface area matches the mesh, without overlap correction.

In [9]:
morph.scale_radii_to_match_mesh(
    mm.mesh, metric="surface_area", account_for_overlaps=False
)

swc_filepath = f"{swc_out_dir}/TS{spine_idx}_mel{mel_tag}_norm.swc"
morph.to_swc_file(swc_filepath)

swc_model = SWCModel.from_swc_file(swc_filepath)
swc_model.print_attributes(node_info=False, edge_info=False)
norm_fig: "go.Figure" = visualize_cable_3d(
    swc_model,
    slider=False,
    title="",
    width=800,
    height=600,
    show_axes=False,
    orientation="horizontal",
    camera=camera,
)
norm_filename = (
    f"{fig_out_dir}/TS{spine_idx}_mel{mel_tag}_" f"morph_norm.pdf"
)
norm_fig.write_image(
    norm_filename,
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
norm_fig.show()

print(swc_filepath)

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=102 reconnections=7 header=11
INFO:swctools.model:SWCModel.from_parse_result records=102 reconnections=7 header=11
INFO:swctools.model:SWCModel.from_swc_file built nodes=102 edges=101 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=101 sides=16 end_caps=False verts=3232 faces=3232
INFO:swctools.geometry:FrustaSet.from_swc_model edges=101 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=101 show_frusta=True show_centroid=True
C:\Users\MainUser\AppData\Local\Temp\ipykernel_8916\4261240504.py:23: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  norm_fig.write_image(
INFO:choreographer.browsers.chromium:Chromium init'ed with kwargs {}
INFO:choreographer.browsers.chromium:Found chromium p

SWCModel: nodes=102, edges=101, components=1, cycles=0, branch_points=11, roots=1, leaves=15, self_loops=0, density=0.0196


INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmp1uckzm22/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido.kaleido:Got 7872
INFO:kaleido.kaleido:Reloading tab 7872 before return.
INFO:kaleido.kaleido:Putting tab 7872 back (queue size: 0).
INFO:kaleido.kaleido:Waiting for all cleanups to finish.
INFO:kaleido.kaleido:Exiting Kaleido.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:kaleido.kaleido:Cancelling tasks.
INFO:kaleido.kaleido:Exiting Kaleido/Choreo.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:kaleido.kaleido:Cancelling tasks.

../../data/swc/current/TS1_qst0.5_mcst5/TS1_mel133.77_norm.swc


### Validation with area fit

In [10]:
validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 2378 vertices, 4788 faces
INFO:mascaf.validation:  Skeleton: 376 nodes, 383 edges
INFO:mascaf.validation:  MorphologyGraph: 95 nodes, 101 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       70083740.2341
INFO:mascaf.validation:---- Morphology volume: 89028600.5561
INFO:mascaf.validation:---- Ratio:             1.2703
INFO:mascaf.validation:---- Error:             18944860.3220
INFO:mascaf.validation:---- Relative error:    27.03%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         3124839.6923
INFO:mascaf.validation:---- Morphology area:   3124839.6923
INFO:mascaf.validation:---- Ratio:             1.0000
INFO:mascaf.validation:---- Error:             -0.0000
INFO:mascaf.validation:---- Relative error:    -0.00%
INFO:mascaf.valid

{'n_outside_nodes': 0, 'n_crossing_edges': 10, 'n_fully_outside_edges': 0}

## Morphology versus skeleton

Overlay the normalized morphology (translucent) with the original skeleton points.

In [11]:
skel_pointset = skeleton.to_point_set()

vs_fig: "go.Figure" = visualize_cable_3d(
    swc_model,
    opacity=0.2,
    title="",
    width=800,
    height=600,
    show_axes=False,
    orientation="horizontal",
    camera=camera,
    point_set=skel_pointset,
    point_color="red",
    point_size=model_length * 0.0015,
)
vs_filename = (
    f"{fig_out_dir}/TS{spine_idx}_mel{mel_tag}_"
    f"morph_vs_skel.pdf"
)
vs_fig.write_image(
    vs_filename,
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
vs_fig.show()

INFO:swctools.geometry:batch_spheres count=-1 stacks=6 slices=12 verts=23312 faces=45120
INFO:swctools.geometry:PointSet.from_points n=376 base_radius=1.0 stacks=6 slices=12
INFO:swctools.geometry:batch_frusta count=101 sides=16 end_caps=False verts=3232 faces=3232
INFO:swctools.geometry:FrustaSet.from_swc_model edges=101 sides=16 end_caps=False
INFO:swctools.geometry:batch_spheres count=376 stacks=6 slices=12 verts=23312 faces=45120
INFO:swctools.geometry:PointSet.scaled radius_scale=4.065675620614607
INFO:swctools.viz:plot_model slider=False frusta=101 show_frusta=True show_centroid=True
C:\Users\MainUser\AppData\Local\Temp\ipykernel_8916\2569096071.py:20: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  vs_fig.write_image(
INFO:choreographer.browsers.chromium:Chromium init'ed with kwargs {}
INFO:choreographer.browsers.chromium:Found chromium path: C:\Program Files

In [12]:
visualize_mesh_cable_3d(mm, skeleton, morph, eye_scale=0.6)

INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:swctools.geometry:batch_frusta count=101 sides=16 end_caps=False verts=3232 faces=3232
INFO:swctools.geometry:FrustaSet.from_swc_model edges=101 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=101 show_frusta=True show_centroid=True
